# Chapitre 11 - Lab : capture des changements avec le décodage logique de PostgreSQL

Le cours 3 a implémenté une CDC jouet par déclencheurs sur SQLite. Les outils réels
(Debezium et consorts) lisent plutôt le **journal d'écriture anticipée** (WAL) de la base,
décodé en changements logiques. Ce lab utilise directement ce mécanisme dans
PostgreSQL 16 : un **slot de réplication logique** avec le module de sortie
`test_decoding`, interrogé en SQL (`pg_logical_slot_get_changes`). On y retrouve, sur un
vrai système, les notions du cours : ordre de validation, position dans le journal (LSN),
système dérivé qui suit la base, instantané initial cohérent, et retard d'un consommateur.

**Démarrer et arrêter les services** (depuis la racine du dépôt) :

```bash
docker compose -f docker/streams/compose.yml -p ddia_streams up -d --wait
# ... exécuter le notebook ...
docker compose -f docker/streams/compose.yml -p ddia_streams down -v
```

PostgreSQL écoute sur le port **11232** de l'hôte (utilisateur et mot de passe `ddia`, base
`flux`), avec `wal_level=logical`.

**Plan.**

1. Un slot de réplication logique : le journal des changements en SQL.
2. L'ordre du journal est l'ordre de validation.
3. Ce que contient un changement : `REPLICA IDENTITY`.
4. Un index dérivé qui suit la base, face à la double écriture.
5. L'instantané initial cohérent.
6. Un slot retient le WAL : le retard du consommateur a un coût.

In [1]:
import re
import threading
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psycopg

from ddia import streams as st
from ddia.timelines import SpaceTime
from ddia.plotting import (AMBER, GRAY, GREEN, INDIGO, LIGHT, ROSE, SKY, SLATE, TEAL, VIOLET,
                           annotate_bars, finish, set_style, subtitle)

set_style()
plt.rcParams["axes.titlepad"] = 16
pd.set_option("display.max_colwidth", 200)
DSN = "host=localhost port=11232 user=ddia password=ddia dbname=flux connect_timeout=5"

def sql(requete, params=None):
    with psycopg.connect(DSN, autocommit=True) as c:
        cur = c.execute(requete, params)
        return cur.fetchall() if cur.description else None

print(sql("SELECT version()")[0][0].split(",")[0], "; wal_level =", sql("SHOW wal_level")[0][0])
for (slot,) in sql("SELECT slot_name FROM pg_replication_slots"):
    sql("SELECT pg_drop_replication_slot(%s)", (slot,))            # notebook relançable
sql("DROP TABLE IF EXISTS produits")
sql("CREATE TABLE produits (id INTEGER PRIMARY KEY, nom TEXT, description TEXT, prix NUMERIC)")

PostgreSQL 16.14 (Debian 16.14-1.pgdg13+1) on x86_64-pc-linux-gnu ; wal_level = logical
